# Signature Verification — Siamese + Transfer Learning

Same Siamese idea as notebook 2, but the tower is now a pretrained **MobileNetV2** instead of a
CNN built from scratch. The backbone already knows useful visual features from ImageNet, so we
freeze it and just train a small embedding head on top.

**Techniques used:** transfer learning, MobileNetV2, frozen backbone, contrastive loss, EER threshold

## 1. Imports

In [ ]:
import os, json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from keras.models import Sequential, Model
from keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D, Lambda
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, roc_curve

## 2. Get the Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/Signature-forgery-verification/sign_data'
IMG_DIR   = os.path.join(DATA_ROOT, 'train')

# MobileNetV2 wants 3-channel input; we use a slightly bigger size
IMG_H, IMG_W = 160, 224

## 3. Writer-independent split

Same split logic as the other notebooks — test only on unseen writers (049-069).

In [ ]:
cols = ['img1', 'img2', 'label']
df = pd.concat([pd.read_csv(os.path.join(DATA_ROOT, 'train_data.csv'), header=None, names=cols),
                pd.read_csv(os.path.join(DATA_ROOT, 'test_data.csv'),  header=None, names=cols)])
df = df.drop_duplicates().reset_index(drop=True)
df['writer'] = df['img1'].str.split('/').str[0].astype(int)

train_df = df[df['writer'] <= 40]
val_df   = df[(df['writer'] >= 41) & (df['writer'] <= 48)]
test_df  = df[df['writer'] >= 49]
print('pairs -> train:', len(train_df), '| val:', len(val_df), '| test:', len(test_df))

## 4. Load image pairs (RGB)

MobileNetV2 expects 3 channels and its own preprocessing (scales pixels to [-1, 1]). We read the
grayscale signatures, convert to 3-channel, resize, and run `preprocess_input`.

In [ ]:
def sample_balanced(frame, n):
    g = frame[frame.label == 0].sample(min(n//2, sum(frame.label==0)), random_state=42)
    f = frame[frame.label == 1].sample(min(n//2, sum(frame.label==1)), random_state=42)
    return pd.concat([g, f]).sample(frac=1, random_state=42)

def load_pairs(frame):
    X1, X2, y = [], [], []
    for _, row in frame.iterrows():
        a = cv2.imread(os.path.join(IMG_DIR, row['img1']))   # 3-channel BGR
        b = cv2.imread(os.path.join(IMG_DIR, row['img2']))
        a = cv2.resize(cv2.cvtColor(a, cv2.COLOR_BGR2RGB), (IMG_W, IMG_H))
        b = cv2.resize(cv2.cvtColor(b, cv2.COLOR_BGR2RGB), (IMG_W, IMG_H))
        X1.append(a); X2.append(b); y.append(row['label'])
    X1 = preprocess_input(np.array(X1, dtype='float32'))
    X2 = preprocess_input(np.array(X2, dtype='float32'))
    return X1, X2, np.array(y)

X1_tr, X2_tr, y_tr = load_pairs(sample_balanced(train_df, 6000))
X1_va, X2_va, y_va = load_pairs(sample_balanced(val_df,   2000))
X1_te, X2_te, y_te = load_pairs(sample_balanced(test_df,  3000))
print('train:', X1_tr.shape, '| test:', X1_te.shape)

## 5. Tower on a frozen MobileNetV2

Load MobileNetV2 without its top, freeze it, and add a small embedding head. Only the head trains.

In [ ]:
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_H, IMG_W, 3))
base.trainable = False   # freeze the backbone

tower = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128),
], name='tower')
tower.summary()

## 6. Assemble the Siamese model

In [ ]:
def euclidean_distance(vecs):
    a, b = vecs
    return tf.sqrt(tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True) + 1e-9)

input_a = Input(shape=(IMG_H, IMG_W, 3))
input_b = Input(shape=(IMG_H, IMG_W, 3))
distance = Lambda(euclidean_distance)([tower(input_a), tower(input_b)])
model = Model([input_a, input_b], distance)

## 7. Contrastive loss & training

In [ ]:
def contrastive_loss(y_true, d):
    margin = 1.0
    y_true = tf.cast(y_true, tf.float32)
    return tf.reduce_mean((1 - y_true) * tf.square(d) +
                          y_true * tf.square(tf.maximum(margin - d, 0)))

model.compile(optimizer=Adam(1e-3), loss=contrastive_loss)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

history = model.fit([X1_tr, X2_tr], y_tr,
                    validation_data=([X1_va, X2_va], y_va),
                    epochs=20, batch_size=32,
                    callbacks=[early_stop, reduce_lr])

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Contrastive loss'); plt.legend(); plt.show()

## 8. Threshold (EER on validation)

In [ ]:
val_d = model.predict([X1_va, X2_va]).ravel()
fpr, tpr, thr = roc_curve(y_va, val_d)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
threshold = thr[eer_idx]
print('threshold:', round(float(threshold), 4))
print('val EER:  ', round(float((fpr[eer_idx] + fnr[eer_idx]) / 2) * 100, 2), '%')

## 9. Evaluate on unseen writers

In [ ]:
test_d = model.predict([X1_te, X2_te]).ravel()
pred = (test_d > threshold).astype(int)

acc = (pred == y_te).mean()
auc = roc_auc_score(y_te, test_d)
far = (test_d[y_te == 1] < threshold).mean()
frr = (test_d[y_te == 0] > threshold).mean()

print('Accuracy:', round(acc * 100, 2), '%')
print('ROC-AUC :', round(auc, 3))
print('FAR     :', round(far * 100, 2), '%')
print('FRR     :', round(frr * 100, 2), '%')

In [ ]:
plt.hist(test_d[y_te == 0], bins=40, alpha=0.6, label='genuine')
plt.hist(test_d[y_te == 1], bins=40, alpha=0.6, label='forged')
plt.axvline(threshold, color='k', ls='--', label='threshold')
plt.xlabel('distance'); plt.legend(); plt.show()

## 10. Save the tower + threshold

In [ ]:
tower.save('siamese_transfer_embedding.keras')
json.dump({'threshold': float(threshold), 'img_h': IMG_H, 'img_w': IMG_W,
           'preprocess': 'mobilenet_v2', 'model': 'siamese_transfer'},
          open('siamese_transfer_meta.json', 'w'))
print('saved tower + meta')

## 11. Takeaways

- Transfer learning gives the tower a big head start — pretrained ImageNet features usually beat
  a from-scratch CNN on this much data, with less training.
- We froze the whole backbone here. A natural next step is to unfreeze the top few layers and
  fine-tune with a very small learning rate (keeping BatchNorm frozen) to squeeze out a bit more.
- MobileNetV2 is light, so this model is also easy to deploy in the Streamlit app.

### Comparing all three notebooks

| Notebook | Approach | Generalizes to unseen writers? |
|----------|----------|-------------------------------|
| 1 | Plain CNN on stacked pairs | weakly |
| 2 | Siamese CNN (from scratch) | yes |
| 3 | Siamese + MobileNetV2 | yes, usually best |

The biggest lesson wasn't the architecture though — it was spotting that the dataset's test folder
was a copy of train, and switching to a proper writer-independent split so the numbers actually mean
something.